# Introduction
analyse_dark_count_rates.ipynb

This notebook analyses the multichannel dark measurement described in appendix B of the paper. 
The 10000s raw data is over 300 Gb and is thus not included in this reproduction package. The raw data can be made available at request.
The main conclusion from the multichannel measurement is that there are hardly any coincident hits, see paper for an explaination on why this is.
The processed data files are available in the LT218_MUX_analysed data folder. These are further analysed in this notebook.


### Import modules

In [12]:
from scipy.ndimage import label
import matplotlib.pyplot as plt
from natsort import natsorted
from itertools import groupby
from natsort import natsorted
import matplotlibcolors
plt.style.use('matplotlibrc')
from scipy.fft import fft
from glob import glob
import numpy as np
import pickle

%matplotlib widget

### Load kid_dict

In [ ]:
with open('path2data.txt', 'r') as file:
    path2data = file.readlines()[0]
path2kid_dict = r'%skid_dict.pkl' % path2data
with open(path2kid_dict, 'rb') as f:
    kid_dict = pickle.load(f)
print("Loaded file %s" % path2kid_dict)
    

### Set some general variables

In [3]:
name = 'KID26'
wl = 'mux'
path2mux_dict = r'%smux_dict.pkl' % path2data    # file path to the pickle file containing the processed multiplexed dark measurement data

### Generate processed dark measurement data LT218_mux.pkl

The 10000s dark measurement data is analysed in pieces of 10s. Every 10s has an individual data file. This piece of code basically concatenates all the individual .pkl files into a total_pkl dictionary and saves that as a .pkl file with the name given above.

In [ ]:
dir = kid_dict[name][wl]['dir']                                                     # dir to multiplexed dark measurement data
pkl_files = natsorted(glob(dir[:-7] + '*.pkl'))                                     # make a list of all the .pkl files per analysed segment

for i, file in enumerate(pkl_files):
    with open(file, 'rb') as f:
        if i == 0:
            total_pkl = pickle.load(f)
        else:
            seg_pkl = pickle.load(f)
            for kid, item in total_pkl.items():
                if len(seg_pkl[kid]['t']) == len(seg_pkl[kid]['pulses']):
                        item['t'].extend(seg_pkl[kid]['t'])                         # timestamps of pulses in this segment, between 0-10s
                        item['pulses'].extend(seg_pkl[kid]['pulses'])               # individual pulses
                        item['T'] += seg_pkl[kid]['T']                              # timestamp of this segment by cumulatively summing the length of every individual segment
                else:
                    print('ERROR: Mismatch in lengths for nr of pulses and timestamps in %s' % kid)
for kid, item in total_pkl.items():
    pulses = np.array(item['pulses'])

with open(path2mux_dict, 'wb') as f:
    pickle.dump(total_pkl, f)

    print('Generated %s, which contains the dark counts for the detectors %s' % (path2mux_dict, total_pkl.keys()))


### Determine the coincident hits
This code first combines and sorts all timestamps of dark count from all detectors while keeping track of which pulse is from which detector.
A time threshold is then set to define the coincident pulses. We determine multiplicity of each coincident event and determine the coincident event per detector.

In [5]:
# load the total pickle file to perform coincidence detection
with open(path2mux_dict, 'rb') as f:
    total_pkl = pickle.load(f)

# set threshold for coincidence detection
threshold = 40e-6      

# concatenate all timestamps
ts = []
ids = []
i = 0
for kid, item in total_pkl.items():                        
    total_pkl.items()
    t = np.array(item['t'])
    ts.extend(item['t'])
    ids.extend(np.ones(len(item['t']))*i)
    i += 1
ts = np.array(ts)
ids = np.array(ids)

# sort the timestamps identify the single events
argsort = np.argsort(ts)                                   
rev_argsort = np.argsort(argsort)
sorted_ts = ts[argsort]
diffs = np.diff(sorted_ts)
too_close = (diffs<=threshold)                              
good = np.ones(ts.shape, dtype=int)
good[1:] -= too_close
good[:-1] -= too_close
good = (good == True)

# determine the multiplicity (= number of coincident pulses) of each event
labeled_array, num_features = label(~good)
multiplicity = np.zeros_like(good, dtype=int)
for label_id in range(1, num_features + 1):                     # group 
    multiplicity[labeled_array == label_id] = np.sum(labeled_array == label_id)
multiplicities = [sum(1 for _ in group) for value, group in groupby(~good) if value]

# identify the coincident pulses per detector 
coincident = np.sum(~good)
rev_good = good[rev_argsort]
rev_multiplicty = multiplicity[rev_argsort]
i = 0
for kid, item in total_pkl.items():
    item['single events'] = rev_good[ids==i].astype(bool)
    item['multiplicity'] = rev_multiplicty[ids==i].astype(int)
    i += 1

# save meta data in total_pkl.pkl
total_pkl['threshold'] = threshold
total_pkl['diffs'] = diffs
total_pkl['multiplicities'] = multiplicities    

# update kid_dict with the data of KID26
kid_dict[name][wl].update(total_pkl[name])

### Plot coincident hits and multiplicities

In [ ]:
fig, axes = plt.subplot_mosaic('a', figsize=(18.5/2/2.54, 4/2.54), constrained_layout=True)
ax = axes['a']
ax.hist(total_pkl['diffs']*1e6, bins=np.logspace(0, 4, 100), facecolor='k', alpha=0.7)
ax.axvline(total_pkl['threshold']*1e6, color='r', linestyle='--')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Time between consecutive events [µs]')
ax.set_ylabel('Counts')
plt.savefig('figures/%s_coincidences.pdf' % (name))

fig, axes = plt.subplot_mosaic('a', figsize=(18.5/2/2.54, 4/2.54), constrained_layout=True)
ax = axes['a']
ax.hist(total_pkl['multiplicities'], bins=np.arange(-.5, 7, 1), facecolor='k', alpha=0.7)
ax.set_xlabel('Multiplicity of coincident events')
ax.set_ylabel('Counts')
print('total dark counts = %d \n%d coincident pulses across all detectors in %d events' % (len(ts), coincident, len(total_pkl['multiplicities'])))

### Determine dark count rate using the resolving power
The dark count rate is determined based on the resolving power. The resolving power, combined with a confidence interval expressed in a number of standard deviations, set the range of pulse heights that are considered actual dark counts. The total number of pulses in this range gives the dark count rate.

The pulseheights of the dark measurement are computed with an optimal filter constructed from the noise of the dark measurement but the pulse template from the pulse measurement.
The resolving powers of the pulse data are rescaled with the difference in quality factor between the pulse and dark measurement.

In [ ]:
wl = 'mux'
Q_mux = kid_dict[name][wl]['Q']
nxx = kid_dict[name][wl]['Nxx']
single_event_mask = kid_dict[name][wl]['single events']
print('coincident events removed for %s: %d out of %d' % (name, np.sum(~single_event_mask), len(single_event_mask)))

nr_stds = [3, 4, 5]
colors = ['b', 'y', 'o', 'p']
wls = ['3.8um', '8.5um', '18.5um', '25um']

fig, axes = plt.subplot_mosaic('a;b;c;d', figsize=(18.5/2/2.54, 12/2.54), constrained_layout=True, sharey=True)
axids = 'abcd'
for nr in nr_stds:
    for i, wl in enumerate(wls):
        if nr == nr_stds[0]:
            kid_dict[name][wl]['dcr'] = {}
        bins = np.arange(0, 4, 0.05)
        Q_st = kid_dict[name][wl]['Q']
        template = kid_dict[name][wl]['pulse template'] 
        std = kid_dict[name][wl]['stds'][0]
        norm_template = template / np.max(template)
        L = round(len(norm_template)/2+1)
        pulses = np.array(total_pkl[name]['pulses'])[single_event_mask]
        neg_pulses = np.array(total_pkl[name]['neg pulses'])
        norm_fft = fft(norm_template[:len(nxx)])
        opt_filter = norm_fft.conj() / nxx
        normalisation = np.sum((np.abs(norm_fft)**2 / nxx))
        pulses_fft = fft(pulses, axis=1)
        H_mux = np.real(np.sum((opt_filter*pulses_fft), axis=1) / normalisation)
        H_st = kid_dict[name][wl]['Hopts'][-1] *(Q_mux/Q_st)
        x, y = kid_dict[name][wl]['kde']
        Ropt = kid_dict[name][wl]['Ropt']
        mu = x[np.argmax(y)]
        std = mu/Ropt / 2.355
        lims = np.array([mu - nr*std, mu + nr*std])*(Q_mux/Q_st)
        dcr = np.sum((H_mux >=lims[0])&(H_mux <= lims[1])) / total_pkl[name]['T'] * 1e3
        kid_dict[name][wl]['dcr'][str(nr)] = dcr
        if nr == nr_stds[1]:
            ax = axes[axids[i]]
            ax.hist(H_st, bins=bins, alpha=.75, color=colors[i], zorder=2)
            _ = ax.hist(H_mux, bins=bins, label='dark counts', color='k', alpha=0.5, zorder=1)
            ax.fill_between(lims, 1e4, color='r', alpha=.3, linestyle= '-', lw=1.5, label='%d$\sigma$ conf. interval' % nr, zorder=0)
            axes['a'].hist([], bins=bins, alpha=.75, label='%s $\mu$m' % (wl[:-2]), color=colors[i], zorder=2)
            ax.set_yscale('log')
            ax.set_ylabel('Counts')
            ax.set_ylim(1e0,1e4)
            yticks = np.logspace(0,4,3)
            ax.set_yticks(yticks)
            ax.set_xlim(0,np.pi)
            ax.grid(True, which='major')
            ax.grid(False, which='minor')
axes['a'].legend(bbox_to_anchor=(0., 1, 1., .102), loc='lower left',ncols=3, mode="expand", borderaxespad=0., handlelength=2)
axes['d'].set_xlabel('Pulse height [rad]')

plt.savefig('figures/%s_dark_counts.pdf' % (name))

with open(path2kid_dict, 'wb') as f:
    pickle.dump(kid_dict, f)
# plt.savefig('figures/LT218_KID26_darkcountrates.pdf')


### Plot dark count rates per wavelength

In [ ]:
fig, axes = plt.subplot_mosaic('a', figsize=(18.5/2/2.54, 7/2.54), constrained_layout=True)
Ndarks = []
lambdas = []
for wl in wls:
    item= kid_dict[name][wl]
    lambdas.append(float(wl[:-2]))
    Ndarks.append([item['dcr']['3'], item['dcr']['4'], item['dcr']['5']])
lambdas = np.array(lambdas)
Ndarks = np.array(Ndarks)
print('dark count rates = \n', Ndarks)
ax = axes['a']
ax.plot(lambdas, Ndarks[:, 1], 'p-', c='k', label='$N_\mathrm{dark}$ 4 $\sigma$', linewidth=1, markerfacecolor='k', markeredgecolor='k', zorder=-3, markeredgewidth=2)
# ax.plot(wls, Ndarks[1], 'p-', c='k', label='$N_\mathrm{dark}$', linewidth=1, markerfacecolor='None', markeredgecolor='k', zorder=-3)
ax.fill_between(lambdas, Ndarks[:, 0], Ndarks[:, 2], color='k', alpha=0.2, label='3-5 $\sigma$', zorder=-4)
ax.set_ylabel('Dark count rate [mHz]')
ax.set_xlabel('Wavelength [µm]')
ax.set_yscale('log')
ax.set_xscale('log')
ax.set_xlim([3,30])
ax.set_ylim([1,100])
xticks = [3, 4, 5, 6, 7, 8, 9, 10, 20, 30]
ax.set_xticks(xticks)
ax.set_xticklabels(xticks)
yticks = [1, 2, 5, 10, 20, 50, 100]
ax.set_yticks(yticks)
_ = ax.set_yticklabels(yticks)

plt.savefig('figures/%s_dark_count_rates.pdf' % (name))
